# Figurl



In [ ]:
# trial_intervals = np.array([t_out[:n_trials], t_poke[1 : n_trials + 1]]).T
# trial_intervals = trial_intervals
# t_0 = trial_intervals[0][0] - 1
t_0 = trial_df.t_poke.values[5] - 1
plot_interval = np.array([[t_0, t_0 + 1000]])
ind_mark = interval_list_contains_ind(plot_interval, analysis.t)

ind_mark
c_temp = analysis.c[ind_mark]

In [ ]:
stay_df = trial_df[trial_df.n_pre_switch > 0]
switch_df = trial_df[trial_df.n_pre_switch == 0]

stay_intervals = np.array(stay_df[["run_start", "t_poke"]].values)
ind = np.logical_and(
    stay_intervals[:, 0] > plot_interval[0, 0],
    stay_intervals[:, 0] < plot_interval[0, 1],
)
stay_intervals = stay_intervals[ind]

switch_intervals = np.array(switch_df[["run_start", "t_poke"]].values)
ind = np.logical_and(
    switch_intervals[:, 0] > plot_interval[0, 0],
    switch_intervals[:, 0] < plot_interval[0, 1],
)
switch_intervals = switch_intervals[ind]

from spyglass.utils.spikesorting import firing_rate_from_spike_indicator

time_mua = np.arange(np.min(plot_interval), np.max(plot_interval), 0.005)
min_time, max_time = time_mua[[0, -1]]
# spike_times = list([*times_list])
spike_indicator = np.zeros((len(time_mua), len(spikes)))

for ind, times in enumerate(spikes):
    times = times[np.logical_and(times >= min_time, times <= max_time)]
    spike_indicator[:, ind] = np.bincount(
        np.digitize(times, time_mua[1:-1]),
        minlength=time_mua.shape[0],
    )

if spike_indicator.ndim == 1:
    spike_indicator = spike_indicator[:, np.newaxis]

mua_plot = firing_rate_from_spike_indicator(spike_indicator, time_mua, multiunit=True)


# LFP
from spyglass.lfp.v1 import LFPV1

query = LFPV1() & {
    "nwb_file_name": "wilbur20210404_.nwb",
    "target_interval_list_name": "02_r1 noPrePostTrialTimes",
}

lfp_df = query.fetch1_dataframe()
t_lfp = lfp_df.index.values
# t_lfp = t_lfp - first_mark_time
ind_lfp = interval_list_contains_ind(plot_interval, t_lfp)

# Position
from spyglass.position.v1 import TrodesPosV1
from spyglass.common.common_position import IntervalPositionInfo

pos_key = {
    "nwb_file_name": "wilbur20210404_.nwb",
    "interval_list_name": "pos 1 valid times",
    "position_info_param_name": "default",
}
# TrodesPosV1 & key
pos_query = IntervalPositionInfo & pos_key
pos_df = pos_query.fetch1_dataframe()
t_pos = pos_df.index.values
# t_pos = t_pos - first_mark_time
ind_pos = interval_list_contains_ind(plot_interval, t_pos)
speed = pos_df.head_speed.values

In [ ]:
import scipy.signal


def smooth(data, n=5, sigma=None, hamming=False, mode="valid"):
    """smooths data with gaussian kernel of size n"""
    if n % 2 == 0:
        n += 1  # make sure n is odd
    if sigma is None:
        sigma = n / 2
    kernel = gkern(n, sigma)[:, None]
    if hamming:
        n = sigma
        kernel = np.ones((sigma, 1)) / sigma
    if len(data.shape) == 1:
        pad = np.ones(((n - 1) // 2, 1))
        return np.squeeze(
            scipy.signal.convolve2d(
                np.concatenate(
                    [pad * data[:, None][0], data[:, None], pad * data[:, None][-1]],
                    axis=0,
                ),
                kernel,
                mode=mode,
            )
        )
    else:
        pad = np.ones(((n - 1) // 2, data.shape[1]))
        return scipy.signal.convolve2d(
            np.concatenate([pad * data[0], data, pad * data[-1]], axis=0),
            kernel,
            mode=mode,
        )


def gkern(l: int = 5, sig: float = 1.0):
    """
    creates gaussian kernel with side length `l` and a sigma of `sig`
    """
    ax = np.linspace(-(l - 1) / 2.0, (l - 1) / 2.0, l)
    gauss = np.exp(-0.5 * np.square(ax) / np.square(sig))
    return gauss / np.sum(gauss)
    kernel = np.outer(gauss, gauss)

In [ ]:
import sortingview.views as vv

os.environ["KACHERY_API_KEY"] = "g9UmADRrAGfSPs1RRNKd6DtYTao3WQd2"
color_list = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
]

# CONTEXT
# subsample = 10
# context_graph = vv.TimeseriesGraph()
# c_fig = pca.transform(c[ind_mark])
# c_fig = smooth(c_fig, sigma=int(10), hamming=True)
# c_fig = c_fig[::subsample]
# for i in np.arange(
#     8,
# ):
#     context_graph.add_line_series(
#         name=f"context pc: {i}",
#         t=mark_times[ind_mark[::subsample]] / 1000,
#         y=c_fig[:, i].astype(np.float32),
#         width=2,
#         color=color_list[i],
#     )

subsample = 10
context_graph = vv.TimeseriesGraph()
c_fig = analysis.c_pca[ind_mark]
c_fig = smooth(c_fig, sigma=int(10), hamming=True)
c_fig = c_fig[::subsample]
t_fig = analysis.t

for i in np.arange(
    3,
):
    context_graph.add_line_series(
        name=f"context pc: {i}",
        t=t_fig[ind_mark[::subsample]],
        y=c_fig[:, i].astype(np.float32),
        width=2,
        color=color_list[i],
    )


context_graph2 = vv.TimeseriesGraph()
for i in np.arange(3, 7):
    context_graph2.add_line_series(
        name=f"context pc: {i}",
        t=t_fig[ind_mark[::subsample]],
        y=c_fig[:, i].astype(np.float32),
        width=2,
        color=color_list[i],
    )


# LFP
lfp_graph = vv.TimeseriesGraph()
lfp_graph.add_line_series(
    name="lfp",
    t=t_lfp[ind_lfp],
    y=lfp_df.iloc[ind_lfp][0].values,
    width=2,
    color="black",
)

# Speed
speed_graph = vv.TimeseriesGraph()
speed_graph.add_interval_series(
    name="stay trials",
    t_start=stay_intervals[:, 0],
    t_end=stay_intervals[:, 1],
    color="grey",
)
speed_graph.add_interval_series(
    name="switch trials",
    t_start=switch_intervals[:, 0],
    t_end=switch_intervals[:, 1],
    color="firebrick",
)

speed_graph.add_interval_series(
    name="poke_times",
    t_start=all_poke_intervals[:, 0],
    t_end=all_poke_intervals[:, 1],
    color="pink",
)

speed_graph.add_interval_series(
    name="reward_pump",
    t_start=all_pump_intervals[:, 0],
    t_end=all_pump_intervals[:, 1],
    color="cornflowerblue",
)

speed_graph.add_line_series(
    name="speed",
    t=t_pos[ind_pos],
    y=speed[ind_pos].astype(np.float32),
    width=2,
    color="teal",
)

# MUA
mua_graph = vv.TimeseriesGraph()
mua_graph.add_line_series(
    name="mua",
    t=time_mua,
    y=mua_plot.astype(np.float32),
    width=2,
    color="k",
)

# ASSEMBLE
context_panels = [
    vv.LayoutItem(speed_graph, stretch=1),
    vv.LayoutItem(lfp_graph, stretch=1),
    vv.LayoutItem(mua_graph, stretch=1),
    vv.LayoutItem(context_graph, stretch=3),
    vv.LayoutItem(context_graph2, stretch=3),
]

view = vv.Box(
    direction="horizontal",
    show_titles=True,
    height=8,
    items=[
        vv.LayoutItem(
            vv.Box(
                direction="vertical",
                show_titles=True,
                items=context_panels,
            )
        ),
    ],
)
view.url(label="test url")